# Hybrid Fusion: Weighted Score dan RRF

Notebook ini membandingkan dua strategi fusi dengan keluaran komponen yang sama: **Item-CF** dan **TF-IDF CBF**. Parameter fusi dipilih memakai validation warm. Test set tidak digunakan.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Buka notebook dari folder proyek agar folder src/ dan data tersedia.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_project_data
from src.evaluation import build_relevant_items, evaluate_ranking
from src.recommenders import fit_popularity, fit_item_cf, fit_tfidf_cbf, recommend_from_model, recommend_popularity
from src.fusion import weighted_score_fusion, reciprocal_rank_fusion

TOP_N = 200
K = 10
data = load_project_data(PROJECT_ROOT)
train, items = data.train, data.items
candidates, eval_users, validation = data.candidates, data.evaluable_users, data.validation_warm
relevant = build_relevant_items(validation, eval_users, positive_threshold=4)
popularity = fit_popularity(train, positive_threshold=4)
seen = {user: set(group.item_id.astype(str)) for user, group in train.groupby("user_id", sort=False)}
pop_recs = recommend_popularity(eval_users, candidates, popularity, seen, top_n=TOP_N)

print(f"User evaluasi: {len(eval_users):,} | Kandidat: {len(candidates):,} | Target relevan per protokol: {len(relevant):,} user")

User evaluasi: 11,659 | Kandidat: 22,753 | Target relevan per protokol: 11,659 user


## Fit komponen hybrid

Kode model yang sama dipakai untuk menghasilkan ranking Item-CF dan TF-IDF CBF. Implementasi TF-IDF hanya dibuat sekali dan digunakan sebagai komponen hybrid.

Untuk Item-CF dan CBF, item yang memiliki skor nol dipakai sebagai pengisi ranking dengan urutan popularitas. Ini menjaga daftar rekomendasi cukup panjang dan membuat fallback-nya eksplisit.

In [2]:
item_cf = fit_item_cf(train, candidates, top_k=100, positive_threshold=4)
tfidf_cbf = fit_tfidf_cbf(train, items, candidates, text_column="content_text_basic", top_k=100, positive_threshold=4)
cf_recs = recommend_from_model(item_cf, eval_users, candidates, popularity, top_n=TOP_N)
cbf_recs = recommend_from_model(tfidf_cbf, eval_users, candidates, popularity, top_n=TOP_N)

component_results = pd.DataFrame([
    {"Metode": "Item-CF", **evaluate_ranking(cf_recs, relevant, eval_users, k=K)},
    {"Metode": "TF-IDF CBF", **evaluate_ranking(cbf_recs, relevant, eval_users, k=K)},
]).set_index("Metode")[ ["recall@10", "ndcg@10"] ]
component_results

,recall@10,ndcg@10
Metode,,
Item-CF,0.005885,0.003351
TF-IDF CBF,0.012234,0.007492


## Weighted Score Fusion

Skor setiap komponen dinormalisasi per user (min-max pada daftar top-200), lalu digabungkan. Bobot CF dipilih dari validation dengan NDCG@10 sebagai kriteria utama dan Recall@10 sebagai pemecah seri.

In [3]:
weighted_rows = []
weighted_runs = {}
for cf_weight in [i / 10 for i in range(11)]:
    recs = weighted_score_fusion(cf_recs, cbf_recs, eval_users, cf_weight=cf_weight, top_n=TOP_N)
    metrics = evaluate_ranking(recs, relevant, eval_users, k=K)
    weighted_runs[cf_weight] = recs
    weighted_rows.append({"bobot_cf": cf_weight, **metrics})
weighted_results = pd.DataFrame(weighted_rows).drop(columns="users_evaluated").sort_values(["ndcg@10", "recall@10"], ascending=False)
weighted_results

,bobot_cf,recall@10,ndcg@10
3,0.3,0.012942,0.007988
4,0.4,0.012613,0.007762
2,0.2,0.012338,0.007674
1,0.1,0.012541,0.007632
0,0.0,0.012234,0.007492
5,0.5,0.012009,0.007391
6,0.6,0.010800,0.006281
7,0.7,0.008744,0.004995
8,0.8,0.006822,0.004035
9,0.9,0.006253,0.003622


## Reciprocal Rank Fusion (RRF)

RRF menggabungkan posisi ranking, tanpa mengandalkan skala skor mentah. Konstanta `rrf_k` dipilih di validation dengan kriteria yang sama.

In [4]:
rrf_rows = []
rrf_runs = {}
for rrf_k in [10, 20, 40, 60, 80, 100]:
    recs = reciprocal_rank_fusion(cf_recs, cbf_recs, eval_users, rrf_k=rrf_k, top_n=TOP_N)
    metrics = evaluate_ranking(recs, relevant, eval_users, k=K)
    rrf_runs[rrf_k] = recs
    rrf_rows.append({"rrf_k": rrf_k, **metrics})
rrf_results = pd.DataFrame(rrf_rows).drop(columns="users_evaluated").sort_values(["ndcg@10", "recall@10"], ascending=False)
rrf_results

,rrf_k,recall@10,ndcg@10
5,100,0.014263,0.009075
3,60,0.013594,0.008765
4,80,0.013397,0.008689
2,40,0.012813,0.008350
1,20,0.012394,0.007884
0,10,0.011615,0.007203


## Ringkasan perbandingan

Tabel berikut menunjukkan baseline komponen dan strategi fusi terbaik pada validation warm. Ini tetap hasil validasi; evaluasi test akhir dilakukan terpisah setelah parameter dibekukan dan file test tersedia.

In [5]:
best_weight = weighted_results.iloc[0]["bobot_cf"]
best_rrf_k = int(rrf_results.iloc[0]["rrf_k"])
best_weighted_metrics = weighted_results.iloc[0].drop(labels="bobot_cf").to_dict()
best_rrf_metrics = rrf_results.iloc[0].drop(labels="rrf_k").to_dict()
summary = pd.DataFrame([
    {"Metode": "Popularity", **evaluate_ranking(pop_recs, relevant, eval_users, k=K)},
    {"Metode": "Item-CF", **evaluate_ranking(cf_recs, relevant, eval_users, k=K)},
    {"Metode": "TF-IDF CBF", **evaluate_ranking(cbf_recs, relevant, eval_users, k=K)},
    {"Metode": f"Weighted Score (CF={best_weight:.1f})", **best_weighted_metrics},
    {"Metode": f"RRF (k={best_rrf_k})", **best_rrf_metrics},
]).set_index("Metode")[ ["recall@10", "ndcg@10"] ]
summary.sort_values("ndcg@10", ascending=False)

,recall@10,ndcg@10
Metode,,
Popularity,0.013499,0.009222
RRF (k=100),0.014263,0.009075
Weighted Score (CF=0.3),0.012942,0.007988
TF-IDF CBF,0.012234,0.007492
Item-CF,0.005885,0.003351
